In [ ]:
from flask import Flask, render_template, jsonify, request
import math

app = Flask(__name__)

# System Constants & State
state = {
    "x": 100.0, "y": 100.0,      # System Position x(t)
    "vx": 0.0, "vy": 0.0,        # Velocity Vectors
    "target_x": 0.0, "target_y": 0.0, # System Origin {0,0}
    "mass": 1.0,
    "dt": 0.1,                   # Time Step
    "damping": 0.95              # Friction/Damping
}

# PID / Control Parameters
K = {"gain": 10.7, "damp": 7.3, "int": 3.0}

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/update', methods=['POST'])
def update():
    data = request.json
    dist_x = data.get('dist_x', 0) # Keyboard 'Disturbance' X
    dist_y = data.get('dist_y', 0) # Keyboard 'Disturbance' Y
    
    # 1. Calculate e(t) - Deviation from Stable Point
    error_x = state["target_x"] - state["x"]
    error_y = state["target_y"] - state["y"]
    
    # 2. Control Force (Fc) - Simplified Proportional Logic
    # In a full PID, we'd track the integral and derivative here
    fc_x = error_x * (K["gain"] / 100)
    fc_y = error_y * (K["gain"] / 100)
    
    # 3. Sum of Forces: F_net = Fc + F_dist
    f_net_x = fc_x + dist_x
    f_net_y = fc_y + dist_y
    
    # 4. Physics Integration (Euler)
    accel_x = f_net_x / state["mass"]
    accel_y = f_net_y / state["mass"]
    
    state["vx"] = (state["vx"] + accel_x) * state["damping"]
    state["vy"] = (state["vy"] + accel_y) * state["damping"]
    
    state["x"] += state["vx"]
    state["y"] += state["vy"]
    
    return jsonify({
        "x": state["x"], "y": state["y"],
        "fc": math.sqrt(fc_x**2 + fc_y**2),
        "error": math.sqrt(error_x**2 + error_y**2)
    })

if __name__ == '__main__':
    app.run(debug=True, port=8011, use_reloader = False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:8011
Press CTRL+C to quit
127.0.0.1 - - [13/Apr/2026 14:01:06] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:01:07] "POST /update HTTP/1.1" 200 -
127.0.0.